# SIPM: Convex Example

A minimal convex example with a single linear constraint `x[0] <= 0`.

In [2]:
import math
import torch
from sipm import make_sampler, SIPMConfig, StochasticInexactPenaltyMethod, constant_schedule

def sampler(batch_size: int):
    zeta = torch.zeros(batch_size, 1)
    xi = torch.zeros(batch_size, 1)
    return zeta, xi

def constraint_map(xi_batch):
    batch = xi_batch.shape[0]
    n = 4
    a = torch.zeros(batch, 1, n)
    a[:, 0, 0] = 1.0
    b = torch.zeros(batch, 1)
    return a, b

c = torch.tensor([2.0, -1.0, 0.5, 1.5])
def objective(x, zeta_batch):
    batch = zeta_batch.shape[0]
    loss = 0.5 * (x[1:] - c[1:]).pow(2).sum()
    return loss.repeat(batch)

def step_size(k: int) -> float:
    if k == 0:
        return 1.0 / (math.log(2.0) ** 2)
    return 1.0 / ((math.log(k + 2.0) ** 2) * math.sqrt(k))

def penalty_weight(k: int) -> float:
    return math.log(k + 1.0)

def delta(k: int) -> float:
    if k == 0:
        return 1.0
    return 1.0 / k

cfg = SIPMConfig(
    num_iters=500,
    step_size=step_size,
    penalty_weight=penalty_weight,
    delta=delta,
    iterate_weight=step_size,
    batch_size=1,
)

solver = StochasticInexactPenaltyMethod(
    objective=objective,
    constraint_map=constraint_map,
    sampler=make_sampler(sampler),
    config=cfg,
)

x0 = torch.zeros(4)
x_bar, history = solver.run(x0)
print("x_bar:", x_bar)
print("final penalty:", history["penalty"][-1])

x_bar: tensor([-0.2862, -1.2741,  0.6371,  1.9112])
final penalty: 0.0
